<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/HEADSWAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 1. Install dependencies
!pip install -q "torchao>=0.16.0"
!pip install -q diffusers transformers accelerate peft safetensors pillow opencv-python-headless

from IPython.display import clear_output
clear_output()
print("✅ Installed")

✅ Installed


In [ ]:
#@title 2. Load Qwen-Image + BFS Head Swap LoRA
import torch
from diffusers import DiffusionPipeline
from diffusers.utils import load_image
from huggingface_hub import hf_hub_download
import gc

# Clear memory
torch.cuda.empty_cache()
gc.collect()

print("Loading base model (this takes 1-2 minutes)...")

# Remove device_map so that enable_model_cpu_offload works properly
pipe = DiffusionPipeline.from_pretrained(
    "Qwen/Qwen-Image-2.1",
    torch_dtype=torch.bfloat16
)

print("Loading BFS Head Swap LoRA...")
pipe.load_lora_weights(
    "Alissonerdx/BFS-Best-Face-Swap",
    weight_name="bfs_head_v1.1_qwen_2.1.safetensors"
)

# Apply efficient offloading and optimizations
pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()

print("✅ Model + LoRA loaded")

Loading base model (this takes 1-2 minutes)...


/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading BFS Head Swap LoRA...


Loading adapter weights from state_dict led to unexpected keys found in the model: transformer_blocks.0.img_mlp.gate_up.lora_A.weight, transformer_blocks.0.img_mlp.gate_up.lora_B.weight, transformer_blocks.1.img_mlp.gate_up.lora_A.weight, transformer_blocks.1.img_mlp.gate_up.lora_B.weight, transformer_blocks.2.img_mlp.gate_up.lora_A.weight, transformer_blocks.2.img_mlp.gate_up.lora_B.weight, transformer_blocks.3.img_mlp.gate_up.lora_A.weight, transformer_blocks.3.img_mlp.gate_up.lora_B.weight, transformer_blocks.4.img_mlp.gate_up.lora_A.weight, transformer_blocks.4.img_mlp.gate_up.lora_B.weight, transformer_blocks.5.img_mlp.gate_up.lora_A.weight, transformer_blocks.5.img_mlp.gate_up.lora_B.weight, transformer_blocks.6.img_mlp.gate_up.lora_A.weight, transformer_blocks.6.img_mlp.gate_up.lora_B.weight, transformer_blocks.7.img_mlp.gate_up.lora_A.weight, transformer_blocks.7.img_mlp.gate_up.lora_B.weight, transformer_blocks.8.img_mlp.gate_up.lora_A.weight, transformer_blocks.8.img_mlp.gate

✅ Model + LoRA loaded


In [ ]:
#@title 3. Create Head Swap server
%%writefile /content/headswap_server.py
import os, io, base64, traceback
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from PIL import Image
import torch
from diffusers import DiffusionPipeline

print("Loading Qwen-Image 2.1 + BFS Head Swap LoRA...", flush=True)

pipe = DiffusionPipeline.from_pretrained(
    "Qwen/Qwen-Image-2.1",
    torch_dtype=torch.bfloat16
)
pipe.load_lora_weights(
    "Alissonerdx/BFS-Best-Face-Swap",
    weight_name="bfs_head_v1.1_qwen_2.1.safetensors"
)
pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()

print("Model + LoRA loaded.", flush=True)

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class HeadSwapRequest(BaseModel):
    body: str          # base64 data URL
    head: str          # base64 data URL
    prompt: str = ""

def dataurl_to_pil(data_url: str) -> Image.Image:
    if "," in data_url:
        data_url = data_url.split(",", 1)[1]
    img_bytes = base64.b64decode(data_url)
    return Image.open(io.BytesIO(img_bytes)).convert("RGB")

def pil_to_dataurl(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=95)
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f"data:image/jpeg;base64,{b64}"

@app.get("/")
def root():
    return {"status": "ok", "service": "headswap"}

@app.post("/headswap")
def headswap(req: HeadSwapRequest):
    try:
        body_img = dataurl_to_pil(req.body)
        head_img = dataurl_to_pil(req.head)

        prompt = req.prompt.strip() or (
            "head_swap: start with <image1> as the base image, keeping its lighting, environment, and background. "
            "remove the head from <image1> completely and replace it with the head from <image2>, "
            "strictly preserving the hair, eye color, nose structure from <image2>. "
            "copy the direction of the eye, head rotation, micro expressions from <image1>, "
            "high quality, sharp details, 4k"
        )

        print("Running head swap...", flush=True)
        result = pipe(
            prompt=prompt,
            image=[body_img, head_img],   # body first, head second
            num_inference_steps=12,
            guidance_scale=1.0,
            height=1024,
            width=1024
        ).images[0]

        return {"image": pil_to_dataurl(result)}

    except Exception as e:
        traceback.print_exc()
        return {"error": str(e)}

if __name__ == "__main__":
    import uvicorn
    print("Server listening on port 8000.", flush=True)
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

Writing /content/headswap_server.py


In [ ]:
#@title 4. Start server + Cloudflare tunnel
import os, re, sys, time, subprocess, threading

# Install server dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "python-multipart"], check=True)

SERVER_SCRIPT = "/content/headswap_server.py"
SERVER_LOG = "/content/headswap_server.log"
PORT = 8000
CLOUDFLARED = "/usr/local/bin/cloudflared"

# Kill any previous server / tunnel
subprocess.run(["pkill", "-f", "headswap_server.py"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(1)

# Download cloudflared if needed
if not os.path.exists(CLOUDFLARED):
    print("Downloading cloudflared...")
    subprocess.run([
        "wget", "-q", "-O", CLOUDFLARED,
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    ], check=True)
    os.chmod(CLOUDFLARED, 0o755)

# Start the FastAPI server
print("Starting Head Swap server...")
with open(SERVER_LOG, "w") as log:
    server_proc = subprocess.Popen(
        [sys.executable, SERVER_SCRIPT],
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

# Wait until the model is loaded
print("Waiting for model to load (this can take 1–2 minutes)...")
for _ in range(180):
    time.sleep(2)
    if os.path.exists(SERVER_LOG):
        with open(SERVER_LOG) as f:
            content = f.read()
            if "Model + LoRA loaded" in content or "Server listening" in content:
                print("✅ Model loaded")
                break
else:
    print("⚠️ Model took too long. Check the log below.")
    !tail -30 /content/headswap_server.log

# Start Cloudflare tunnel
print("Starting Cloudflare tunnel...")
tunnel_proc = subprocess.Popen(
    [CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True
)

TUNNEL_URL = None
t0 = time.time()
while time.time() - t0 < 60:
    line = tunnel_proc.stdout.readline()
    if not line:
        break
    m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
    if m:
        TUNNEL_URL = m.group(0)
        break

if TUNNEL_URL:
    print("\n" + "="*60)
    print(f"READY: {TUNNEL_URL}")
    print("="*60)
    print("\n→ Copy the URL above and paste it into the Head Swap tab on your website.")
else:
    print("❌ Could not get Cloudflare URL. Try re-running this cell.")